In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

results_path = r'C:\Users\niyat\Desktop\pr\ML-Bias-Fairness\results'

print("Libraries loaded successfully")
print("Starting preprocessing for all 3 datasets...")

Libraries loaded successfully
Starting preprocessing for all 3 datasets...


In [2]:
# ══ ADULT DATASET PREPROCESSING ══════════════════

# Reload dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = [
    'age','workclass','fnlwgt','education','education-num',
    'marital-status','occupation','relationship','race',
    'sex','capital-gain','capital-loss','hours-per-week',
    'native-country','income'
]
df_adult = pd.read_csv(url, names=columns,
                        na_values=' ?',
                        skipinitialspace=True)

# Step 1 — Drop missing values
df_adult = df_adult.dropna()
print(f"After dropping missing: {df_adult.shape}")

# Step 2 — Encode target variable
df_adult['income_binary'] = (df_adult['income'] == '>50K').astype(int)

# Step 3 — Encode protected characteristics
df_adult['sex_binary'] = (df_adult['sex'] == 'Male').astype(int)

# Step 4 — Encode categorical features
cat_cols = ['workclass','education','marital-status',
            'occupation','relationship','race','native-country']
le = LabelEncoder()
for col in cat_cols:
    df_adult[col] = le.fit_transform(df_adult[col].astype(str))

# Step 5 — Drop original columns no longer needed
df_adult = df_adult.drop(['income','sex','fnlwgt'], axis=1)

# Step 6 — Split features and target
X_adult = df_adult.drop('income_binary', axis=1)
y_adult = df_adult['income_binary']

# Step 7 — Train/test split
X_train_adult, X_test_adult, y_train_adult, y_test_adult = train_test_split(
    X_adult, y_adult, test_size=0.2, random_state=42, stratify=y_adult)

# Step 8 — Scale numerical features
scaler = StandardScaler()
X_train_adult = pd.DataFrame(scaler.fit_transform(X_train_adult),
                               columns=X_adult.columns)
X_test_adult = pd.DataFrame(scaler.transform(X_test_adult),
                              columns=X_adult.columns)

print(f"Training set: {X_train_adult.shape}")
print(f"Test set:     {X_test_adult.shape}")
print(f"Target balance: {y_train_adult.mean()*100:.1f}% positive class")
print("✅ Adult dataset preprocessing complete")

After dropping missing: (32561, 15)
Training set: (26048, 13)
Test set:     (6513, 13)
Target balance: 24.1% positive class
✅ Adult dataset preprocessing complete


In [3]:
# ══ GERMAN CREDIT PREPROCESSING ══════════════════

url_german = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
col_names = [
    'status','duration','credit_history','purpose','credit_amount',
    'savings','employment','installment_rate','personal_status_sex',
    'other_debtors','residence','property','age','other_installments',
    'housing','existing_credits','job','dependents','telephone',
    'foreign_worker','target'
]
df_german = pd.read_csv(url_german, sep=' ', header=None, names=col_names)

# Step 1 — Encode gender
gender_map = {'A91':'Male','A92':'Female','A93':'Male',
              'A94':'Male','A95':'Female'}
df_german['gender_binary'] = df_german['personal_status_sex'].map(gender_map)
df_german['gender_binary'] = (df_german['gender_binary']=='Male').astype(int)

# Step 2 — Encode target (1=Good, 2=Bad → 0=Good, 1=Bad)
df_german['target_binary'] = (df_german['target'] == 2).astype(int)

# Step 3 — Encode categorical features
cat_cols_g = ['status','credit_history','purpose','savings',
              'employment','personal_status_sex','other_debtors',
              'property','other_installments','housing','job',
              'telephone','foreign_worker']
le = LabelEncoder()
for col in cat_cols_g:
    df_german[col] = le.fit_transform(df_german[col].astype(str))

# Step 4 — Drop original target
df_german = df_german.drop('target', axis=1)

# Step 5 — Split
X_german = df_german.drop('target_binary', axis=1)
y_german = df_german['target_binary']

X_train_german, X_test_german, y_train_german, y_test_german = train_test_split(
    X_german, y_german, test_size=0.2, random_state=42, stratify=y_german)

# Step 6 — Scale
scaler_g = StandardScaler()
X_train_german = pd.DataFrame(scaler_g.fit_transform(X_train_german),
                                columns=X_german.columns)
X_test_german = pd.DataFrame(scaler_g.transform(X_test_german),
                               columns=X_german.columns)

print(f"Training set: {X_train_german.shape}")
print(f"Test set:     {X_test_german.shape}")
print("✅ German Credit preprocessing complete")

Training set: (800, 21)
Test set:     (200, 21)
✅ German Credit preprocessing complete


In [4]:
# ══ LAW SCHOOL PREPROCESSING ═════════════════════

df_law = pd.read_csv(
    r'C:\Users\niyat\Desktop\pr\ML-Bias-Fairness\data\law_school\law_dataset.csv'
)

# Step 1 — Drop missing values
df_law = df_law.dropna(subset=['pass_bar','male','racetxt','lsat','ugpa'])

# Step 2 — Encode race
le_race = LabelEncoder()
df_law['race_encoded'] = le_race.fit_transform(df_law['racetxt'].astype(str))

# Step 3 — Select features
features = ['lsat','ugpa','zfygpa','zgpa','fulltime',
            'fam_inc','male','race_encoded','tier']
df_law_clean = df_law[features + ['pass_bar']].dropna()

X_law = df_law_clean[features]
y_law = df_law_clean['pass_bar'].astype(int)

# Step 4 — Split
X_train_law, X_test_law, y_train_law, y_test_law = train_test_split(
    X_law, y_law, test_size=0.2, random_state=42, stratify=y_law)

# Step 5 — Scale
scaler_l = StandardScaler()
X_train_law = pd.DataFrame(scaler_l.fit_transform(X_train_law),
                             columns=X_law.columns)
X_test_law = pd.DataFrame(scaler_l.transform(X_test_law),
                            columns=X_law.columns)

print(f"Training set: {X_train_law.shape}")
print(f"Test set:     {X_test_law.shape}")
print(f"Target balance: {y_train_law.mean()*100:.1f}% pass rate")
print("✅ Law School preprocessing complete")

print("\n" + "="*50)
print("ALL 3 DATASETS PREPROCESSED AND READY FOR MODELLING")
print("="*50)

Training set: (14953, 9)
Test set:     (3739, 9)
Target balance: 90.2% pass rate
✅ Law School preprocessing complete

ALL 3 DATASETS PREPROCESSED AND READY FOR MODELLING
